# NFP — Synthetic Self-Test

End-to-end verification of `RecNFP` on fully synthetic data:
1. Build a 2-D Siemens-star phantom (`proj`) and a Gaussian probe (`prb`).
2. Forward-simulate diffraction patterns via `gen_sqrt_data`.
3. Run iterative reconstruction (BH) and check convergence.

In [ ]:
import numpy as np
import cupy as cp
from mpi4py import MPI
from types import SimpleNamespace
from holotomocupy.rec_nfp_mpi import RecNFP
from holotomocupy.utils import mshow, mshow_complex

## Acquisition Parameters

In [ ]:
n      = 384          # detector / probe size (pixels)
ntheta = 16           # number of scan positions
nobj   = 400          # object size (= 3n//2, allows for sample drift)

energy                  = 17.1          # keV
detector_pixelsize      = 1.4760147601476e-6 * 8   # m (binned)
focustodetectordistance = 1.217         # m
z1                      = 5.110e-3      # m  (sample–focus distance)

## Phantom Object — Siemens Star

A 2-D complex phantom with Gaussian smoothing; real part encodes δ (phase), imaginary part encodes β (absorption) with δ/β = 29.

In [ ]:
delta_beta = 29   # δ/β ratio

def siemens_star(nobj, step_deg=15):
    """Return a (nobj, nobj) float32 Siemens-star mask."""
    def rotate_pts(pts, ang, center):
        c, s = np.cos(ang), np.sin(ang)
        R = np.array([[c, -s], [s, c]], dtype=np.float32)
        return (pts - center) @ R.T + center

    def tri_mask(X, Y, tri):
        (x1,y1),(x2,y2),(x3,y3) = tri
        d = (y2-y3)*(x1-x3) + (x3-x2)*(y1-y3)
        if d == 0: return np.zeros_like(X, dtype=bool)
        a = ((y2-y3)*(X-x3) + (x3-x2)*(Y-y3)) / d
        b = ((y3-y1)*(X-x3) + (x1-x3)*(Y-y3)) / d
        return (a>=0) & (b>=0) & (1-a-b>=0)

    tri0 = np.array([
        (1.5*nobj//16, nobj//2 - nobj//32),
        (1.5*nobj//16, nobj//2 + nobj//32),
        (nobj//2 - nobj//128, nobj//2),
    ], dtype=np.float32)
    yy, xx  = np.mgrid[0:nobj, 0:nobj]
    center  = np.array([nobj/2, nobj/2], dtype=np.float32)
    star    = np.zeros((nobj, nobj), dtype=np.float32)
    for deg in range(0, 360, step_deg):
        star += tri_mask(xx, yy, rotate_pts(tri0, np.deg2rad(deg), center)).astype(np.float32)
    star /= star.max() or 1.0
    return star

def gen_proj(nobj, delta_beta):
    star = cp.array(siemens_star(nobj))
    # smooth with Gaussian in Fourier space
    v  = cp.arange(-nobj//2, nobj//2, dtype='float32') / nobj
    vx, vy = cp.meshgrid(v, v)
    g  = cp.fft.fftshift(cp.exp(-8*(vx**2 + vy**2)))
    star = cp.fft.ifft2(cp.fft.fft2(star) * g).real
    # scale delta to phase range [0, π/4]; beta = delta/delta_beta
    star = star / star.max() * (np.pi / 4)
    # complex: real = delta projection, imag = beta projection
    return (-star + 1j * star / delta_beta).astype('complex64')

proj_gt = gen_proj(nobj, delta_beta)
mshow(proj_gt.real, True)

## Probe — loaded from ID16A tiff files

In [ ]:
from scipy.fft import fft2, ifft2, fftshift
from holotomocupy.utils import *
import scipy.ndimage as ndimage


!wget -nc https://g-110014.fd635.8443.data.globus.org/holotomocupy/examples_synthetic/data/prb_id16a/prb_abs_2048.tiff -P data/prb_id16a
!wget -nc https://g-110014.fd635.8443.data.globus.org/holotomocupy/examples_synthetic/data/prb_id16a/prb_phase_2048.tiff -P data/prb_id16a

prb_abs   = read_tiff('data/prb_id16a/prb_abs_2048.tiff')[:1]    # take 1 distance
prb_phase = read_tiff('data/prb_id16a/prb_phase_2048.tiff')[:1]
prb = (prb_abs * np.exp(1j * prb_phase)).astype('complex64')

# crop to n×n centred patch
prb = prb[:, prb.shape[1]//2-n//2:prb.shape[1]//2+n//2,
              prb.shape[2]//2-n//2:prb.shape[2]//2+n//2]

# apply mild Gaussian filter in Fourier space
v = np.arange(-n//2, n//2, dtype='float32') / n
vx, vy = np.meshgrid(v, v, indexing='ij')
filt = fftshift(np.exp(-4.0 * (vx**2 + vy**2)).astype('float32'))
prb = ifft2(fft2(prb) * filt).astype('complex64')

# normalise amplitude to 1 (single distance, squeeze to 2D)
prb = prb[0]
prb /= np.mean(np.abs(prb))
prb_gt = cp.array(prb)
mshow_complex(prb_gt, True)

## Positions — random sub-pixel shifts

In [ ]:
rng     = np.random.default_rng(10)
pos_gt  = (30 * (rng.random((ntheta, 2)) - 0.5)).astype('float32')   # true positions
pos_err = (     (rng.random((ntheta, 2)) - 0.5)).astype('float32')   # initial guess error

## Initialise RecNFP

In [ ]:
args = SimpleNamespace(
    energy                  = energy,
    detector_pixelsize      = detector_pixelsize,
    focustodetectordistance = focustodetectordistance,
    z1                      = z1,
    ntheta                  = ntheta,
    nz                      = n,
    n                       = n,
    nzobj                   = nobj,
    nobj                    = nobj,
    obj_dtype               = 'complex64',
    rho                     = [1, 2, 0.1],   # [proj, prb, pos]
    niter                   = 257,
    nchunk                  = 8,
    vis_step                = 32,
    err_step                = 32,
    start_iter              = 0,
    comm                    = MPI.COMM_WORLD,
)

cl = RecNFP(args)

## Generate Synthetic Data

In [ ]:
cl.vars['proj'][:] = proj_gt
cl.vars['prb'][:]  = prb_gt
cl.vars['pos'][:]  = cp.array(pos_gt[cl.st_theta:cl.end_theta])

cl.gen_sqrt_data(cl.vars, cl.data)

mshow(cl.data[0], True)

## Reconstruction

Reset variables to initial guess (flat object, flat probe, noisy positions) and run BH.

In [ ]:
cl.vars['proj'][:] = 0
cl.vars['prb'][:]  = 1
cl.vars['pos'][:]  = cp.array((pos_gt + pos_err)[cl.st_theta:cl.end_theta])

cl.BH()

## Results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes[0,0].imshow(proj_gt.real.get(), cmap='gray');       axes[0,0].set_title('GT proj (δ)')
axes[0,1].imshow(proj_gt.imag.get(), cmap='gray');       axes[0,1].set_title('GT proj (β)')
axes[1,0].imshow(cl.vars['proj'].real.get(), cmap='gray'); axes[1,0].set_title('Rec proj (δ)')
axes[1,1].imshow(cl.vars['proj'].imag.get(), cmap='gray'); axes[1,1].set_title('Rec proj (β)')
plt.tight_layout(); plt.show()

print('Convergence table:')
print(cl.table)